# COM804 - Classificando o UrbanSound8k com CNNs
## Treino do modelo Baseline


# Importando tudo

In [18]:
# !!Panic Button
# Deletar checkpoints
# !rm -rf checkpoints
# !mkdir checkpoints

!ls
# import os
# os._exit(0)

checkpoints  data  ykk_repo


In [1]:
from IPython.display import clear_output
from IPython.display import FileLink
# !pip install --upgrade ipython
# %load_ext autoreload
# %autoreload 2

!git clone https://github.com/y7kko/COM804-Classifier.git ykk_repo
!git -C ./ykk_repo/ pull
!pip install git+https://github.com/soundata/soundata.git
!pip install torch-audiomentations
!pip install git+https://github.com/iver56/audiomentations.git


clear_output()
import sys
import os

sys.path.append('./ykk_repo/')
os.environ['YKK_NOCHDIR'] = '1'
os.environ['US8K_PATH'] = './data/urbansound8k'

import ykk
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import optuna
from ykk.neuralnet.dataset import urban8k
urban8k(root_path=os.environ['US8K_PATH']).download_dataset()
import time
import copy
from torchaudio import transforms
from pathlib import Path
Path('./checkpoints/').mkdir(parents=True, exist_ok=True)
Path('./featcache/').mkdir(parents=True, exist_ok=True)
print('Done, check console for more details')
import pickle, copy

_t0= None
_t1 = None

def tic():
    global _t0
    _t0 = time.time()

def toc():
    global _t1,_t0
    _t1 = time.time()
    dt =_t1 - _t0
    _t0,_t1 = None,None
    return dt

Done, check console for more details


# Declarando

## Dataset importer

In [2]:
fs = 44100
transform = nn.Sequential(
    transforms.MelSpectrogram(fs,n_fft=1024,hop_length=1024,n_mels=64,),
    transforms.AmplitudeToDB(stype="power", top_db=80)
)

In [3]:
import os, pickle

def get_data(val_fold, return_dataloader=True, cache_feats=True):
    """Splits dataset"""
    global fs
    global transform

    train_fld = [f for f in range(1, 11) if f != val_fold]
    test_fld = val_fold + 1 if val_fold < 10 else 1
    train_fld = [f for f in train_fld if f != test_fld]
    
    valid_data = urban8k(augmentation= False, fs=fs, filter_folds = val_fold)
    test_data  = urban8k(augmentation= False, fs=fs, filter_folds = test_fld)
    train_data = urban8k(augmentation= True,  fs=fs, filter_folds = train_fld)

    for _data in [train_data, valid_data, test_data]:
        _data.set_transform(transform)
        if cache_feats:
            _data.cache_features(augmentation_sets=1)

    # np.random.seed(0)
    # train_data.cache_signals()

    train_data.masks =[
            transforms.TimeMasking(26, True,.3),
            transforms.FrequencyMasking(14,True)
        ]

    # if cache_validation:
    #     valid_data.cache_features()


    if return_dataloader:
        g = torch.Generator().manual_seed(42)
        train_dl = DataLoader(train_data, shuffle=True, batch_size= 64, pin_memory=True,generator=g)
        valid_dl = DataLoader(valid_data, batch_size= 32)
        test_dl = DataLoader(test_data, batch_size= 32)
        return train_dl, valid_dl, test_dl
    else:
        return train_data, valid_data, test_data

## Parâmetros
Foi utilizado o MFCC transform para realizar extração das features. Embora alguns estudos apontem melhores resultados com STFT ou log-mel, o modelo foi treinado na CPU, e a MFCC oferece dados com menos dimensões.

- `fs`: Frequência de amostragem do sinal
- `mel_kw`: Parâmetros do espectrograma
- `n_mfcc`: número de componentes de frequência por frame de tempo

Nota: para simplificar, o tamanho da input layer foi calculado realizando a extração de features, mas há forma analítica de calcular.

In [4]:
_d = urban8k(fs=fs)
_d.set_transform(transform)
input_shape = list((_d[0][0]).shape)
print(f'input shape: {input_shape} ({np.prod(input_shape)})')

input shape: [1, 64, 130] (8320)


## O modelo
Pode-se interpretar o MFCC como uma imagem, portanto utilizou-se uma CNN 2D. 

In [5]:
class NeuralNetwork(nn.Module):
    def __init__(self,dropout=.5,filter_size=3,**kwargs):
        """Inspirado na CNN6 (Kong, 2020)
        
        """
        # 
        # conv -> batchnorm -> relu -> pool
        super().__init__()
        self.sequential = nn.Sequential(

            # Conv1
            nn.Conv2d(1, 64, filter_size, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            # nn.MaxPool2d(2),
            
            nn.Conv2d(64, 64, filter_size, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),

            nn.AvgPool2d(2),
            nn.Dropout(dropout),
            
            # Conv2
            nn.Conv2d(64, 128, filter_size, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            
            # nn.MaxPool2d(2),
            nn.Conv2d(128, 128, filter_size, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            
            nn.AvgPool2d(2),
            nn.Dropout(dropout),
            
            # Conv3
            nn.Conv2d(128, 256, filter_size, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),

            nn.Conv2d(256, 256, filter_size, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            
            nn.AvgPool2d(2),
            nn.Dropout(dropout),
            # Conv4
            nn.Conv2d(256, 512, filter_size, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(),

            nn.Conv2d(512, 512, filter_size, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(),
            nn.AvgPool2d(2),
            nn.Dropout(dropout),

            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(512, 10)
        )
        self.apply(self._init_weights)

    def _init_weights(self, m): 
        if isinstance(m, nn.Linear): #He
            nn.init.kaiming_uniform_(m.weight, nonlinearity='relu')
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Conv2d):
            nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            if m.bias is not None:
                nn.init.zeros_(m.bias)


    def forward(self, x):
        return self.sequential(x)

In [6]:
sum(p.numel() for p in NeuralNetwork().parameters() if p.requires_grad)

4955850

## Trainer


In [6]:
from sklearn.metrics import f1_score
from tqdm.auto import tqdm
class Trainer():
    def __init__(self,model,loss_fn,optimizer,valid_loss_delta=1E-4,device=None):
        if device is None:
            self.device = self.get_best_device()
        else:
            self.device = device
        print(f'device: {self.device}')
        self.model = model
        self.optimizer =optimizer
        self.loss_fn = loss_fn
        self.valid_loss, self.train_loss = [], []
        self.accuracy, self.f1_score = [], []
        self.epoch = 0
        self.early_stop_flag = False
        self.best_accuracy = 0
        self.best_valid_loss = 0
        self.best_epoch = 0
        self.test_stats = {}
        

    def train(self,train_data:DataLoader):
        size = len(train_data.dataset)
        self.model = self.model.to(self.device)
        self.model.train()
        running_loss,seen = 0.0, 0
        for batch, (X, y) in enumerate(tqdm(train_data, desc="Treino")):
            X, y = self._dev_encode(X,y)
            # Compute prediction error
            pred = self.model(X)
            loss = self.loss_fn(pred, y)

            # Backpropagation
            loss.backward()
            self.optimizer.step()
            self.optimizer.zero_grad()

            running_loss += loss.item()*len(X)
            seen += len(X)
            if batch % 50 == 0:
                loss, current = loss.item(), (batch + 1) * len(X)
                tqdm.write(f"\t loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")
        self.train_loss.append(running_loss/seen)
        self.epoch +=1
        print(f"\t (TRAIN) loss: {running_loss/seen:>7f}")
        return running_loss/seen

    
    @torch.no_grad()
    def validate(self, test_data):
        size = len(test_data.dataset)
        self.model.eval()
        valid_loss, accuracy = 0.0, 0
    
        all_preds, all_labels = [], []
        for X, y in test_data:
            X, y = self._dev_encode(X, y)
    
            logits = self.model(X)                       # logits crus
            valid_loss += self.loss_fn(logits, y).item() * len(X)
            preds = logits.argmax(1)
            accuracy += (preds == y).sum().item()
    
            all_preds.append(preds.cpu())
            all_labels.append(y.cpu())
    
        valid_loss /= size
        accuracy /= size
    
        all_preds  = torch.cat(all_preds).numpy()
        all_labels = torch.cat(all_labels).numpy()
        f1 = f1_score(all_labels, all_preds, average='macro')
    
        self.valid_loss.append(valid_loss)
        self.accuracy.append(accuracy * 100)
        self.f1_score.append(f1)
    
        print(f"\t (VALID) acc: {100*accuracy:>0.1f}%, Avg loss: {valid_loss:>8f}, f1: {f1:.4f}\n")
        return valid_loss, accuracy, f1

    @torch.no_grad()
    def test(self, test_data,return_raw=False):
        size = len(test_data.dataset)
        self.model.eval()
        valid_loss, accuracy = 0.0, 0
    
        all_preds, all_labels = [], []
        for X, y in test_data:
            X, y = self._dev_encode(X, y)
    
            logits = self.model(X)                       # logits crus
            valid_loss += self.loss_fn(logits, y).item() * len(X)
            preds = logits.argmax(1)
            accuracy += (preds == y).sum().item()
    
            all_preds.append(preds.cpu())
            all_labels.append(y.cpu())
    
        valid_loss /= size
        accuracy /= size

        all_preds  = torch.cat(all_preds).numpy()
        all_labels = torch.cat(all_labels).numpy()
        
        f1 = f1_score(all_labels, all_preds, average='macro')
        self.test_stats['accuracy'] = accuracy
        self.test_stats['loss'] = valid_loss
        self.test_stats['f1'] = f1
        print(f"\t (TEST) data acc: {100*accuracy:>0.1f}%, Avg loss: {valid_loss:>8f}, f1: {f1:.4f}\n")

        if return_raw:
            return all_preds,all_labels
        
        return valid_loss, accuracy, f1

    
    def _dev_encode(self,*args):
        return tuple(arg.to(self.device) for arg in args)

    def save_checkpoint(self,fname):
        if fname is None:
            return
    
        checkpoint = {
            'epoch': self.epoch,
            'model_state_dict': self.model.state_dict(),
            'optimizer_state_dict': self.optimizer.state_dict(),
            'validation_loss': self.valid_loss,
            'training_loss': self.train_loss,
            'accuracy':self.accuracy,
            'f1_score':self.f1_score,
            'early_stop_flag':self.early_stop_flag,
            'best_valid_loss': self.best_valid_loss,
            'best_accuracy': self.best_accuracy,
            'best_epoch': self.best_epoch,
            'test_stats': self.test_stats
            }
        torch.save(checkpoint, fname)

    def load_checkpoint(self, fname,boot_from_best=False):
        checkpoint = torch.load(fname, weights_only=False,map_location='cpu',)
        if boot_from_best:
            # print(checkpoint['best_epoch'])
            _slice = slice(checkpoint['best_epoch'])
            current_epoch = checkpoint['best_epoch']
        else:
            _slice = slice(None)
            current_epoch = checkpoint['epoch']

    
        self.model.load_state_dict(checkpoint['model_state_dict'])
        self.optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        self.model.to(self.device)

        self.epoch = current_epoch
        self.valid_loss = checkpoint['validation_loss'][_slice]
        self.train_loss = checkpoint['training_loss'][_slice]
        self.accuracy = checkpoint['accuracy'][_slice]
        self.f1_score = checkpoint['f1_score'][_slice]
        self.early_stop_flag = checkpoint['early_stop_flag']
        self.best_valid_loss = checkpoint['best_valid_loss']
        self.best_accuracy = checkpoint['best_accuracy'] 
        self.best_epoch = checkpoint['best_epoch']
        self.test_stats = checkpoint['test_stats']

    def is_early_stopped(self,fname):
        checkpoint = torch.load(fname, weights_only=False,map_location='cpu',)
        return checkpoint['early_stop_flag']

    def get_best_device(self,):
        try:
            import torch_xla.core.xla_model as xm
            if len(xm.get_xla_supported_devices(devkind="TPU")) > 0:
                return "xla"
        except Exception:
            pass
            
        if torch.cuda.is_available():
            return "cuda"
        return "cpu"

## Treino de um fold
Para cada fold, é chamado `evaluate_fold`, reponsável por separar o dataset em validação e teste. Alguns hiperparâmetros foram expostos para otimização com optuna, sendo estes:
- dropout
- learning rate
- dimensões das hidden layers

In [7]:

def evaluate_fold(fold,dropout,lr,transform,train_dl,valid_dl,test_dl,max_epochs=50,patience=7,
                  checkpoint_filename=None,model=None,l2=0):
    if model is None:
        raise ValueError('Nao especificou model')
    loss_fn = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr,weight_decay=l2)
    trainer = Trainer(model = model, loss_fn = loss_fn,optimizer=optimizer)
    try:
        trainer.load_checkpoint(checkpoint_filename)
        print('checkpoint loaded!')
    except:
        pass
    if trainer.early_stop_flag:
        print('This fold already converged!')
        best_model = trainer.model
        return trainer.best_valid_loss, trainer.best_accuracy, best_model

    best_state = None
    best_acc, early_stop_cnt, best_val_loss, best_f1 = 0, 0, float('inf'),0
    for epoch in range(trainer.epoch,max_epochs):
        print(f':: (FOLD {fold}) Epoch: {epoch} ({epoch-trainer.best_epoch} since best)')
        epoch_t0 = time.time()
        trainer.train(train_dl)
        val_loss, acc,f1 = trainer.validate(valid_dl)
    
        # EarlyStopper
        if acc > trainer.best_accuracy + 1e-3:
            best_f1 = f1
            trainer.best_epoch = epoch
            trainer.best_valid_loss = val_loss
            trainer.best_accuracy = acc
            early_stop_cnt = 0
            best_state = copy.deepcopy(trainer.model.state_dict())
            
            trainer.save_checkpoint(checkpoint_filename)
        else:
            early_stop_cnt += 1
            if early_stop_cnt >= patience:
                trainer.early_stop_flag = True
                break

        epoch_dt = time.time() - epoch_t0
        print(f"\t{epoch_dt/60:.2f} min ({epoch_dt/3600:.2f} h)")
    trainer.model.load_state_dict(best_state)
    trainer.save_checkpoint(checkpoint_filename)
    best_model = trainer.model
    trainer.test(test_dl)
    return trainer.best_valid_loss, trainer.best_accuracy, best_model

## Treinando os folds

In [8]:
val_losses, val_ac = [], []
dropout = .5
lr = 1E-4
l2 = 1E-3

for fold in range(1,11):
    trainer = Trainer(model = None, loss_fn = None,optimizer=None)
    try:
        if trainer.is_early_stopped(f'./checkpoints/fold_{fold}.pth'):
            print(f'Fold {fold} already converged, skipping this fold')
            continue
        else:
            print('ue n parou?')
    except:
        pass

    model = NeuralNetwork(dropout=dropout,filter_size=3)
    train_dl,valid_dl, test_dl = get_data(fold)
    dataloaders = dict(train_dl=train_dl,valid_dl=valid_dl,test_dl=test_dl)
    vl,vac, _ = evaluate_fold(fold,
                              dropout=dropout, 
                              lr=lr,
                              transform=transform,
                              max_epochs=70,
                              patience = 12,
                              model=model,
                              l2=l2,
                              checkpoint_filename=f'./checkpoints/fold_{fold}.pth',
                              **dataloaders
                             )
    val_losses.append(vl)
    val_ac.append(vac)
    print(f'Fold {fold} trained')

device: cpu
Fold 1 already converged, skipping this fold
device: cpu
Fold 2 already converged, skipping this fold
device: cpu
Fold 3 already converged, skipping this fold
device: cpu
Fold 4 already converged, skipping this fold
device: cpu
Fold 5 already converged, skipping this fold
device: cpu
Fold 6 already converged, skipping this fold
device: cpu
Fold 7 already converged, skipping this fold
device: cpu
Fold 8 already converged, skipping this fold
device: cpu
Fold 9 already converged, skipping this fold
device: cpu
Fold 10 already converged, skipping this fold


# Plotando resultados

In [28]:
import os
import h5py
import numpy as np


hdf5_fname = 'baseline_training_results.h5'
if os.path.exists(hdf5_fname):
    os.remove(hdf5_fname)

def _save(group, name, data, dtype):
    arr = np.asarray(data, dtype=dtype)
    if arr.size == 0:
        print(f'skipping {name}: empty')
        return
    if name in group:
        del group[name]
    group.create_dataset(name, data=arr)

for fold in range(1,11):
    train_dl, valid_dl, test_dl = get_data(fold,cache_feats=False)
    model = NeuralNetwork(dropout=dropout,filter_size=3)
    loss_fn = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1E-3,weight_decay=0)
    trainer = Trainer(model = model, loss_fn = loss_fn,optimizer=optimizer)
    trainer.load_checkpoint(f'./checkpoints/fold_{fold}.pth')
    tic()
    y_pred, y_true = trainer.test(test_dl, return_raw=True)
    print(f':::: Test evaluation of fold {fold} took {toc():.2f} s')
    with h5py.File(hdf5_fname, 'a') as f:
        group = f.require_group(f'fold_{fold}')
        _save(group, 'epochs',          np.arange(0, trainer.epoch),        int)
        _save(group, 'train_loss',      trainer.train_loss,                 float)
        _save(group, 'valid_loss',      trainer.valid_loss,                 float)
        _save(group, 'valid_acc',       trainer.accuracy,                   float)
        _save(group, 'valid_f1',        trainer.f1_score,                   float) 
        _save(group, 'best_valid_loss', trainer.best_valid_loss,            float)
        _save(group, 'best_accuracy',   trainer.best_accuracy,              float)
        _save(group, 'best_epoch',      trainer.best_epoch,                 int)
        _save(group, 'test_loss',       trainer.test_stats['loss'],         float)
        _save(group, 'test_acc',        trainer.test_stats['accuracy'],     float)
        _save(group, 'test_f1',         trainer.test_stats['f1'],           float)
        _save(group, 'test_pred',       y_pred,                             int)
        _save(group, 'test_true',       y_true,                             int)
        print('ok')

device: cpu
	 (TEST) data acc: 57.1%, Avg loss: 1.390304, f1: 0.5342

:::: Test evaluation of fold 1 took 51.13 s
ok
device: cpu
	 (TEST) data acc: 67.0%, Avg loss: 1.750627, f1: 0.6871

:::: Test evaluation of fold 2 took 61.57 s
ok
device: cpu
	 (TEST) data acc: 64.8%, Avg loss: 1.645931, f1: 0.6632

:::: Test evaluation of fold 3 took 78.97 s
ok
device: cpu
	 (TEST) data acc: 78.8%, Avg loss: 0.721097, f1: 0.7875

:::: Test evaluation of fold 4 took 53.11 s
ok
device: cpu
	 (TEST) data acc: 71.6%, Avg loss: 1.701038, f1: 0.7204

:::: Test evaluation of fold 5 took 50.17 s
ok
device: cpu
	 (TEST) data acc: 73.9%, Avg loss: 1.064897, f1: 0.7446

:::: Test evaluation of fold 6 took 49.10 s
ok
device: cpu
	 (TEST) data acc: 63.5%, Avg loss: 1.382265, f1: 0.6531

:::: Test evaluation of fold 7 took 59.77 s
ok
device: cpu
	 (TEST) data acc: 74.0%, Avg loss: 1.225380, f1: 0.7494

:::: Test evaluation of fold 8 took 46.95 s
ok
device: cpu
	 (TEST) data acc: 77.5%, Avg loss: 1.020981, f1: 0.

In [29]:
from IPython.display import FileLink
FileLink(hdf5_fname)

/kaggle/working/baseline_training_results.h5

In [ ]:
import shutil
pasta_origem = './checkpoints'
nome_zip = 'checkpoints_baseline'
shutil.make_archive(nome_zip, 'zip', '/kaggle/working', pasta_origem)
FileLink(f'{nome_zip}.zip')